# Mega Project 1 — Intelligent Underwriting & Automated Credit Decisioning
## Problem 1: Credit Default Prediction — EDA, Feature Engineering, Top-4 Model
## Screening → Top-2 5-Fold CV Champion Selection, SHAP + LIME Explainability,
## Statistical Validation & Financial-Impact Reporting

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
Home Credit wants to predict, at the moment of application, whether a client will
have payment difficulty on the loan (`TARGET = 1`) or not (`TARGET = 0`). This is the
foundational model the rest of Mega Project 1 (Loan Application Approval, Credit Score
Estimation, Repayment Capacity Analysis, Previous Application Outcomes) — and several
other Mega Projects — read their probability-of-default (PD) input from.

### Data used (real, verified against your files before this notebook was written)
- `application_train.csv` — 307,511 real applications, 122 real columns, real TARGET
  default rate 8.07%
- `bureau.csv` — 1,716,428 real external credit-bureau records, aggregated to
  customer level (active-credit count, total debt, overdue history, etc.)

### Hardware-utilization fix (this revision — real root cause, not a hardware limit)
A prior real run of this notebook was observed using far less of the CPU/RAM ceiling
it was configured for. Root cause, found by inspecting the actual code order: the
thread-count environment variables (`OMP_NUM_THREADS`, `OPENBLAS_NUM_THREADS`,
`MKL_NUM_THREADS`, `POLARS_MAX_THREADS`, etc.) were never actually being *set*
anywhere, **and** numpy/polars/pandas/scikit-learn/xgboost/lightgbm/catboost were all
imported at the very top of the file, before the ceiling was even computed — every one
of those libraries reads its own thread-count env var exactly once, at its own import
time, so no library ever saw a real ceiling regardless of what was printed. Combined
with `GradientBoostingClassifier` (no multi-core support at all in scikit-learn) and
`LogisticRegression` (whose `n_jobs` has no effect for binary classification) sitting
in the old 6-model benchmark, this is the concrete, code-level explanation — not a
hardware limitation. Fixed this revision by:
- Importing and calling the shared `src/utils/performance_setup.py` module (HYPER) as
  literally the first executable step, before any BLAS/OpenMP-reading library is
  imported — env vars are now actually set before they are read.
- Explicitly pinning this process's CPU affinity to every detected logical core,
  removing any pre-existing OS-level core restriction the env vars alone cannot fix.
- Dropping GradientBoostingClassifier and LogisticRegression from the model set (see
  below) — both would otherwise run pinned to one thread while every other configured
  thread sits idle.
- Adding Parquet-over-CSV caching (WARP): the first real run still pays the normal CSV
  parse cost, but every run after that reads a much faster cached columnar Parquet
  copy of each raw file instead — a genuine, stated trade-off, not a claim that the
  first run gets faster too.

### Model benchmark redesign: Top-4 screen → Top-2 5-fold CV (reduced from 6 models)
1. **Stage A — screening**: all 4 real candidates (RandomForest, XGBoost, CatBoost,
   LightGBM — chosen for real predictive strength on tabular credit data *and*
   genuine multi-core parallelism) are each fit once on an 80/20 real
   train/validation split.
2. **Stage B — champion selection**: only the top 2 screened candidates are promoted
   to a real 5-fold `StratifiedKFold` CV (roughly half the CV cost of the old
   6-model × 5-fold benchmark). The champion is the higher mean-CV-AUC model of
   those two.
3. **CV Report**: a real per-fold ROC-AUC table + chart for the top-2 models, not
   just their mean/std.
4. **SHAP explainability** (champion model only): `shap.TreeExplainer` on a real
   300-row holdout sample — a mean-|SHAP| bar chart plus a beeswarm detail chart.
5. **LIME explainability** (champion model only): real local explanations for 3
   representative real holdout cases (most-confident correct default flag,
   most-confident correct non-default, a real misclassified application).

### What this notebook does (SOP Stages 1B–6 in one run)
1. Applies the WARP hardware fix above, then loads both real files via a
   Parquet-cached Polars read.
2. Runs real Exploratory Data Analysis & data-quality checks on the raw data before
   feature engineering — missingness, IQR outliers, target correlation, the real
   DAYS_EMPLOYED sentinel-value anomaly — with 3 vivid multicolor chart figures
   (SOP Stage 1B/2).
3. Engineers ~38 features via the shared `src/features/credit_default_features.py`
   module (HYPER standard — built once, imported everywhere; Notebook 03 / Problem 3
   imports this exact same function to reproduce identical features for scoring):
   application fields, bureau-aggregate features, and derived ratios (credit-to-income,
   annuity-to-income, age, years employed).
4. Splits train/holdout (85/15, stratified, seed 42) — encoders and imputers are
   fit on train only, never on holdout (no leakage).
5. Runs the Top-4 screen → Top-2 5-fold CV champion selection described above;
   retrains the champion on full train, evaluates once on the untouched holdout set;
   displays the ROC curve, screening chart, CV benchmark chart, and CV report chart
   inline (vivid multicolor, per the standing chart-style rule).
6. Computes and displays real SHAP + LIME explainability for the champion model.
7. Runs real Statistical Validation (SOP Stage 4): bootstrap 95% CI on holdout
   ROC-AUC, calibration-by-decile, split-half PSI, and an explicit deployment
   readiness verdict.
8. Runs 13 integrity self-checks (fails loudly rather than silently passing bad
   state — including new checks that the CPU thread ceiling was actually applied,
   the screen was really top-4, CV really ran on only 2 models, SHAP values are
   finite, and LIME explanations were really computed), then generates a full
   Stage-5 reporting package — CSV outputs (including model screening, CV report,
   SHAP importances, and LIME explanations as their own files), a colorized Word
   report, an 8-sheet Excel workbook, and an HTML dashboard with 8 live charts (3
   with slicer/filter dropdowns, plus a filterable sampled prediction table) — via
   the shared `src/reporting/report_builder.py` module (HYPER).
9. Saves the champion model + a full run summary (including the real
   performance-config, model-selection, and explainability metadata) to
   `../decision_engine/artifacts/` — overwritten in place every run (idempotent),
   never appended.

### Standing rules this notebook follows
- **Zero-fabrication**: every number below is computed live, this run, from your own
  copy of the real data — nothing is carried over from a prior session.
- **WARP**: resource ceilings capped at 90% RAM / 95% CPU threads (never 100%, a
  safety ceiling — not a floor forced by padding), applied *before* any heavy
  import; CPU affinity pinned; Parquet-over-CSV caching; vectorized Polars
  throughout; `RANDOM_SEED = 42`.
- **HYPER**: shared `src/features/`, `src/reporting/`, and `src/utils/` modules,
  built once, imported here and reused by Notebook 03 (and now every notebook in
  this mega project).
- **Privacy**: this notebook never prints your machine's absolute file paths, since
  notebooks in this suite may be shared publicly (e.g. on GitHub/Kaggle).

### Before you run this
Create `project_config.json` at the `home-credit-enterprise-suite` project root
(one level above every `mega_project_N_.../` folder) with:
```json
{
  "raw_data_dir": "<absolute path to your folder containing application_train.csv, bureau.csv, ...>",
  "random_seed": 42,
  "ram_ceiling_fraction": 0.90,
  "cpu_ceiling_fraction": 0.95
}
```
This file is machine-specific — keep it out of version control (already listed in
the suite's `.gitignore`). On your real machine, `shap` and `lime` must be installed
(`pip install shap lime`) — both are lightweight, pure-Python-plus-compiled-extension
packages with no extra system dependency.

### Verification status
Verified end-to-end on a synthetic fixture matching the real schema via real Jupyter
execution (`jupyter nbconvert --execute`) — 0 errors, all 13 integrity checks passed,
HTML dashboard confirmed rendering charts/filter dropdowns with 0 console errors under
a network-blocked Playwright check, Excel formulas confirmed correct via LibreOffice
headless recalculation, and the Parquet cache confirmed correct (a second read returns
byte-identical data to the original CSV read). **Not yet run against your real data.**


In [ ]:
# ============================================================================
# NOTEBOOK 01 — MEGA PROJECT 3: INTELLIGENT UNDERWRITING & AUTOMATED CREDIT
# DECISIONING | PROBLEM 1: CREDIT DEFAULT PREDICTION
# Business Understanding, EDA, Feature Engineering, Top-4 Model Screening ->
# Top-2 5-Fold CV Champion Selection, SHAP + LIME Explainability, Statistical
# Validation & Financial-Impact Reporting (SOP Stages 1-6 in one run)
# ----------------------------------------------------------------------------
# Zero-fabrication notice: every number this cell prints, plots, or writes to a
# report is computed live, right now, from YOUR OWN copy of the real Kaggle Home
# Credit Default Risk dataset. Nothing here is carried over from a prior run or
# invented. Re-running this cell always overwrites the same output paths
# (idempotent).
#
# HARDWARE-UTILIZATION FIX (this revision): every BLAS/OpenMP thread-count
# environment variable is now set — via the shared, HYPER src/utils/
# performance_setup.py module, not a local duplicate — BEFORE numpy, polars,
# pandas, scikit-learn, xgboost, lightgbm, or catboost are imported anywhere
# below. The PREVIOUS version of this file computed a thread ceiling but never
# actually set any environment variable, AND it imported all of those libraries
# at the very top of the file, before that ceiling was even computed — so no
# library ever saw a real ceiling regardless. Combined with GradientBoosting
# Classifier (no multi-core support at all in scikit-learn) and LogisticRegres-
# sion (n_jobs has no effect for binary classification) sitting in the old
# 6-model benchmark, this is the concrete, code-level explanation for a real
# run observably using a fraction of the CPU/RAM ceiling it was configured for
# — not a hardware limitation. See PERFORMANCE_SETUP_README.md / WARP notes.
# ============================================================================

import os
import sys
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# SECTION 1 — Config + suite-root resolution (stdlib only — no heavy library
# is imported yet, deliberately, so the WARP thread ceiling below can be set
# before any of them read their thread-count environment variables. Never
# print the resolved raw-data path itself: this notebook may be shared
# publicly, e.g. on GitHub/Kaggle).
# ---------------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd()
def _find_suite_root(start: Path = None) -> Path:
    """Locate the home-credit-enterprise-suite project root (the folder containing
    project_config.json), regardless of where this notebook's kernel actually launched
    from. Checked in order, fastest and most explicit first -- deliberately NOT an
    unbounded/recursive filesystem scan (the exact "hangs / looks frozen" risk this
    suite's WARP performance module exists to avoid):
    1. HC_SUITE_ROOT environment variable, if set (see PERFORMANCE_SETUP_README.md)
    2. Walking UPWARD from the working directory (covers: cwd is this notebook's own
       mega_project_.../notebooks/ folder, the normal case when opened in place)
    3. A short list of well-known locations under the home directory (covers: the
       working directory being your home folder itself -- an ANCESTOR of the project,
       not inside it -- which an upward-only search cannot reach)
    """
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory plus "
        "well-known locations under your home folder. Fix: either open this notebook's "
        "own .ipynb file in place (rather than running its code in a fresh kernel "
        "elsewhere), or set an environment variable before launching Jupyter, e.g. on "
        'Windows PowerShell: $env:HC_SUITE_ROOT="C:\\Users\\rnand\\Downloads\\'
        'home-credit-enterprise-suite" -- see PERFORMANCE_SETUP_README.md.'
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

RAW_DIR = Path(CONFIG["raw_data_dir"])
SEED = int(CONFIG.get("random_seed", 42))
RANDOM_SEED = SEED

ARTIFACTS_DIR = SUITE_ROOT / "01_mega_project_1_underwriting_approval" / "decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = SUITE_ROOT / "01_mega_project_1_underwriting_approval" / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR = SUITE_ROOT / "01_mega_project_1_underwriting_approval" / "decision_engine" / "_parquet_cache"

# HYPER standing rule: shared feature-engineering + reporting + performance logic
# lives in src/, imported here rather than duplicated inline -- this is the module
# Notebook 03 (Problem 3: Credit Score Estimation) also imports to reproduce the
# identical feature set for scoring.
sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import (
    configure_performance, pin_cpu_affinity, sklearn_n_jobs, gbm_thread_kwargs,
    threadpool_guard, free_memory, check_ram_headroom, load_csv_cached,
)

# ---------------------------------------------------------------------------
# SECTION 2 — WARP resource ceilings (hard cap, never 100%) — set BEFORE any
# of numpy/polars/pandas/scikit-learn/xgboost/lightgbm/catboost are imported.
# configure_performance() sets OMP_NUM_THREADS / OPENBLAS_NUM_THREADS /
# MKL_NUM_THREADS / NUMEXPR_NUM_THREADS / POLARS_MAX_THREADS (etc.) as real OS
# environment variables — every one of those libraries reads its own copy of
# these exactly once, at its own import/init time, so this call MUST happen
# first. pin_cpu_affinity() additionally pins this process to every detected
# logical core, removing any pre-existing OS-level core restriction the env
# vars alone cannot fix.
# ---------------------------------------------------------------------------
PERF = configure_performance(
    ram_ceiling_fraction=float(CONFIG.get("ram_ceiling_fraction", 0.90)),
    cpu_ceiling_fraction=float(CONFIG.get("cpu_ceiling_fraction", 0.95)),
)
pin_cpu_affinity(PERF)
TOTAL_RAM_GB = PERF["total_ram_gb"]
TOTAL_THREADS = PERF["logical_cores"]
RAM_CEILING_GB = PERF["ram_ceiling_gb"]
CPU_CEILING_THREADS = PERF["n_threads"]

# ---------------------------------------------------------------------------
# SECTION 3 — Heavy-library imports (deliberately AFTER Section 2 above)
# ---------------------------------------------------------------------------
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score, brier_score_loss,
    roc_curve, confusion_matrix,
)
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import pandas as pd
import shap
from lime.lime_tabular import LimeTabularExplainer

np.random.seed(SEED)
T0 = time.time()

from features.credit_default_features import engineer_credit_default_features, engineer_credit_default_features_v2
from reporting.report_builder import (
    write_csv_outputs, build_word_report, build_excel_workbook,
    build_html_dashboard, assumption_ref, VIVID_PALETTE, _palette,
)
# Standing chart-style rule (all problems): vivid multicoloured charts everywhere,
# using the 8-hue CVD-validated categorical palette canonicalized in
# src/reporting/report_builder.py (HYPER -- imported, not redefined per notebook).

REQUIRED_FILES = [
    "application_train.csv", "bureau.csv", "bureau_balance.csv", "previous_application.csv",
    "POS_CASH_balance.csv", "installments_payments.csv", "credit_card_balance.csv",
]
missing = [fn for fn in REQUIRED_FILES if not (RAW_DIR / fn).exists()]
if missing:
    raise FileNotFoundError(f"Missing required raw file(s) in configured raw_data_dir: {missing}")

print(f"[WARP] {TOTAL_RAM_GB:.1f} GB RAM / {TOTAL_THREADS} threads detected -> "
      f"ceiling {RAM_CEILING_GB} GB RAM, {CPU_CEILING_THREADS} threads "
      f"(env vars applied before any heavy import; CPU affinity pinned to all cores)")
print("[DATA] Raw data directory resolved and verified (path withheld from output by design).")
print(f"[SEED] RANDOM_SEED = {SEED} (fixed for full reproducibility)")

# ---------------------------------------------------------------------------
# SECTION 4 — Load real data (WARP: Parquet-over-CSV cache — first run reads
# the real CSV and writes a Parquet cache under decision_engine/_parquet_cache/;
# every run after that reads the much-faster columnar Parquet copy instead.
# Never mutates your raw Kaggle folder.)
#
# v2 FEATURE-SET EXPANSION (this revision): Problem 1's champion model
# previously trained on only 2 of Home Credit's 7 real data tables
# (application_train, bureau). This revision loads all 7 and trains on the
# full, leakage-safe feature set from `engineer_credit_default_features_v2`
# -- a deliberate, measured accuracy fix (see Section 9 below for the real,
# same-split holdout AUC comparison against the old 2-table feature set),
# not a routine addition. Every downstream notebook that reuses this
# champion's PD (Notebook 03/04/05, Mega Project 2's Notebook 01) inherits
# the accuracy improvement automatically since they import the same shared
# `engineer_credit_default_features_v2` function to reproduce these exact
# features at scoring time.
# ---------------------------------------------------------------------------
app = load_csv_cached(RAW_DIR / "application_train.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
bureau = load_csv_cached(RAW_DIR / "bureau.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
bureau_balance = load_csv_cached(RAW_DIR / "bureau_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA"])
previous_application = load_csv_cached(RAW_DIR / "previous_application.csv", PARQUET_CACHE_DIR,
                                        null_values=["", "NA", "XNA", "XAP"])
pos_cash = load_csv_cached(RAW_DIR / "POS_CASH_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
installments = load_csv_cached(RAW_DIR / "installments_payments.csv", PARQUET_CACHE_DIR, null_values=["", "NA"])
credit_card = load_csv_cached(RAW_DIR / "credit_card_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
check_ram_headroom(PERF)

N_APP_RAW = app.height
N_BUREAU_RAW = bureau.height
print(f"[LOAD] application_train: {N_APP_RAW:,} rows x {app.width} cols")
print(f"[LOAD] bureau: {N_BUREAU_RAW:,} rows x {bureau.width} cols")

if "TARGET" not in app.columns:
    raise ValueError("TARGET column missing from application_train.csv — cannot proceed.")

TARGET_RATE = float(app["TARGET"].mean())
print(f"[EDA] Real default rate (TARGET=1): {TARGET_RATE:.4%}")

# ---------------------------------------------------------------------------
# SECTION 5 — Exploratory Data Analysis & Data Quality (SOP Stage 1B/2)
# Real EDA computed on the RAW data, before any cleaning/imputation is applied —
# every chart below shows what your actual data looks like prior to Section 7's
# cleaning steps, not a post-cleaning view dressed up as "before".
# ---------------------------------------------------------------------------
null_counts = app.null_count().to_pandas().T.reset_index()
null_counts.columns = ["column", "n_null"]
null_counts["pct_null"] = null_counts["n_null"] / N_APP_RAW
null_counts = null_counts[null_counts["n_null"] > 0].sort_values("pct_null", ascending=False)
top_missing = null_counts.head(15)
print(f"[EDA] {len(null_counts)} / {app.width} real columns have at least one missing value. "
      f"Top 5 by % missing:")
for _, row in null_counts.head(5).iterrows():
    print(f"  {row['column']}: {row['pct_null']:.2%} ({int(row['n_null']):,} rows)")

target_counts = app["TARGET"].value_counts().sort("TARGET").to_pandas()

N_DAYS_EMPLOYED_ANOMALY = int((app["DAYS_EMPLOYED"] == 365243).sum())
PCT_DAYS_EMPLOYED_ANOMALY = N_DAYS_EMPLOYED_ANOMALY / N_APP_RAW
print(f"[EDA] DAYS_EMPLOYED anomaly (sentinel value 365243, a known Home Credit data-quality "
      f"issue meaning 'not currently employed'): {N_DAYS_EMPLOYED_ANOMALY:,} / {N_APP_RAW:,} rows "
      f"({PCT_DAYS_EMPLOYED_ANOMALY:.2%}). Converted to null before modeling (Section 7).")

income_type_counts = app["NAME_INCOME_TYPE"].value_counts().sort("count", descending=True).to_pandas()

DIST_COLS = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY"]
dist_data = {c: app[c].drop_nulls().to_numpy() for c in DIST_COLS}

OUTLIER_SUMMARY = []
for c in DIST_COLS:
    vals = dist_data[c]
    q1, q3 = np.percentile(vals, [25, 75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((vals < lo) | (vals > hi)).sum())
    OUTLIER_SUMMARY.append({"column": c, "n_outliers_iqr": n_out, "pct_outliers_iqr": n_out / len(vals),
                             "iqr_lower": float(lo), "iqr_upper": float(hi)})
    print(f"[EDA] IQR outliers in {c}: {n_out:,} ({n_out / len(vals):.2%}) outside [{lo:,.0f}, {hi:,.0f}]")

CORR_COLS = [c for c in [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
    "REGION_POPULATION_RELATIVE", "CNT_CHILDREN", "CNT_FAM_MEMBERS",
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "DAYS_BIRTH",
] if c in app.columns]
corr_pdf = app.select(["TARGET"] + CORR_COLS).to_pandas()
target_corr = corr_pdf.corr(numeric_only=True)["TARGET"].drop("TARGET").sort_values()
print(f"[EDA] Real correlation with TARGET (top 3 by |r|): "
      f"{target_corr.abs().sort_values(ascending=False).head(3).round(4).to_dict()}")

# --- EDA Figure 1: Data Quality Overview (2x2, vivid multicolor) -----------
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

axes[0, 0].barh(top_missing["column"][::-1], (top_missing["pct_null"][::-1] * 100),
                 color=_palette(len(top_missing)))
axes[0, 0].set_xlabel("% Missing"); axes[0, 0].set_title(f"Top {len(top_missing)} Columns by Real Missing-Value %")

axes[0, 1].bar(["No Default (0)", "Default (1)"], target_counts["count"].tolist(),
                color=["#0ca30c", "#d03b3b"])  # status palette: good / critical -- semantically apt here
axes[0, 1].set_title(f"Real Target Class Balance (default rate {TARGET_RATE:.2%})")
for i, v in enumerate(target_counts["count"].tolist()):
    axes[0, 1].text(i, v, f"{v:,}", ha="center", va="bottom")

axes[1, 0].bar(["Normal value", "Anomaly (365243)"],
                [N_APP_RAW - N_DAYS_EMPLOYED_ANOMALY, N_DAYS_EMPLOYED_ANOMALY],
                color=[VIVID_PALETTE[0], VIVID_PALETTE[7]])
axes[1, 0].set_title("DAYS_EMPLOYED Sentinel-Value Anomaly (real count)")

top_income_types = income_type_counts.head(6)
axes[1, 1].bar(top_income_types["NAME_INCOME_TYPE"], top_income_types["count"],
                color=_palette(len(top_income_types)))
axes[1, 1].set_title("Real NAME_INCOME_TYPE Distribution")
axes[1, 1].tick_params(axis="x", rotation=30)

plt.tight_layout()
eda_overview_path = REPORTS_DIR / "notebook_01_eda_overview.png"
plt.savefig(eda_overview_path, dpi=110)
plt.show()

# --- EDA Figure 2: Numeric Distributions & IQR Outliers (2x3, vivid) -------
fig, axes = plt.subplots(2, len(DIST_COLS), figsize=(5 * len(DIST_COLS), 8))
for i, c in enumerate(DIST_COLS):
    axes[0, i].hist(dist_data[c], bins=40, color=VIVID_PALETTE[i % len(VIVID_PALETTE)], edgecolor="white")
    axes[0, i].set_title(f"Real Distribution: {c}")
    bx = axes[1, i].boxplot(dist_data[c], vert=False, patch_artist=True)
    bx["boxes"][0].set_facecolor(VIVID_PALETTE[(i + 3) % len(VIVID_PALETTE)])
    n_out = next(o["n_outliers_iqr"] for o in OUTLIER_SUMMARY if o["column"] == c)
    axes[1, i].set_title(f"IQR Outliers: {n_out:,} ({n_out / len(dist_data[c]):.1%})")
plt.tight_layout()
eda_distributions_path = REPORTS_DIR / "notebook_01_eda_distributions.png"
plt.savefig(eda_distributions_path, dpi=110)
plt.show()

# --- EDA Figure 3: Real correlation with TARGET (sign-colored, vivid) ------
fig, ax = plt.subplots(figsize=(9, 5.5))
colors = [VIVID_PALETTE[0] if v >= 0 else VIVID_PALETTE[7] for v in target_corr]
ax.barh(target_corr.index, target_corr.values, color=colors)
ax.axvline(0, color="#898781", linewidth=1)
ax.set_title("Real Pearson Correlation with TARGET\n(blue = positive, red = negative)")
plt.tight_layout()
eda_correlation_path = REPORTS_DIR / "notebook_01_eda_correlation.png"
plt.savefig(eda_correlation_path, dpi=110)
plt.show()

EDA_CHART_PATHS = [eda_overview_path, eda_distributions_path, eda_correlation_path]

# ---------------------------------------------------------------------------
# SECTION 6 — Feature engineering (WARP: vectorized Polars aggregation, no
# Python-level row loops) — via the shared src/features module (HYPER: built
# once, imported everywhere; Notebook 03/04/05 and Mega Project 2's Notebook
# 01 all import this exact same v2 function to reproduce these features
# identically at scoring time). v2 adds bureau_balance, previous_application,
# POS_CASH_balance, installments_payments, and credit_card_balance on top of
# v1's application + bureau fields (see Section 4's v2 note above and this
# module's own docstring for the full leakage-safety reasoning per table).
# v1 (`engineer_credit_default_features`) is retained, unused below, only so
# Section 9's real, same-split accuracy comparison can show the honest
# before/after -- it is not used to train the deployed champion.
# ---------------------------------------------------------------------------
df, NUMERIC_FEATURES, CATEGORICAL_FEATURES = engineer_credit_default_features_v2(
    app, bureau, bureau_balance, previous_application, pos_cash, installments, credit_card
)
FEATURE_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# v1 (2-table) feature frame, computed alongside v2 purely so Section 9 can
# report a real, same-split holdout AUC comparison -- never used for the
# deployed champion itself.
df_v1, NUMERIC_FEATURES_V1, CATEGORICAL_FEATURES_V1 = engineer_credit_default_features(app, bureau)

pdf = df.select(["SK_ID_CURR", "TARGET"] + FEATURE_COLS).to_pandas()
for c in CATEGORICAL_FEATURES:
    # fillna BEFORE casting to category: a NaN inside a pandas Categorical cannot be
    # filled after the fact (fillna requires the value to already be a category), and
    # OrdinalEncoder leaves true NaN un-encoded (neither a known nor "unknown" category)
    pdf[c] = pdf[c].astype(object).fillna("Missing").astype(str).astype("category")
for c in NUMERIC_FEATURES:
    pdf[c] = pdf[c].astype("float32")

print(f"[FEATURES] {len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical "
      f"= {len(FEATURE_COLS)} total features built from real data")

# ---------------------------------------------------------------------------
# SECTION 7 — Train / holdout split (stratified, seeded)
# ---------------------------------------------------------------------------
X = pdf[FEATURE_COLS]
y = pdf["TARGET"].astype(int)
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)
print(f"[SPLIT] train={X_train.shape[0]:,} holdout={X_holdout.shape[0]:,} "
      f"(holdout default rate {y_holdout.mean():.4%})")

# encoders/imputers fit on TRAIN ONLY, applied to both (no leakage)
ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
imputer = SimpleImputer(strategy="median")

X_train_enc = X_train.copy()
X_holdout_enc = X_holdout.copy()
if CATEGORICAL_FEATURES:
    X_train_enc[CATEGORICAL_FEATURES] = ord_enc.fit_transform(X_train[CATEGORICAL_FEATURES].astype(str))
    X_holdout_enc[CATEGORICAL_FEATURES] = ord_enc.transform(X_holdout[CATEGORICAL_FEATURES].astype(str))
X_train_enc[NUMERIC_FEATURES] = imputer.fit_transform(X_train[NUMERIC_FEATURES])
X_holdout_enc[NUMERIC_FEATURES] = imputer.transform(X_holdout[NUMERIC_FEATURES])

# ---------------------------------------------------------------------------
# SECTION 8 — Model candidate screening: TOP 4 real models only (reduced from
# a prior 6-model set). The 4 were chosen for real predictive strength on
# tabular credit data AND genuine multi-core parallelism: GradientBoosting
# Classifier (no n_jobs support at all in scikit-learn's implementation) and
# LogisticRegression (n_jobs has no effect on binary classification) are the
# two models this notebook dropped -- both would otherwise run pinned to a
# single thread while the rest of the configured CPU_CEILING_THREADS threads
# sit idle, a real, code-level cause of low observed CPU utilization during
# training, independent of the underlying hardware.
# ---------------------------------------------------------------------------
def make_candidate_models():
    return {
        "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=8, random_state=SEED,
                                                 n_jobs=CPU_CEILING_THREADS),
        "XGBoost": xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8,
                                      colsample_bytree=0.8, eval_metric="auc", random_state=SEED,
                                      n_jobs=CPU_CEILING_THREADS),
        "CatBoost": cb.CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, random_seed=SEED,
                                           verbose=False, thread_count=CPU_CEILING_THREADS,
                                           train_dir=str(ARTIFACTS_DIR / "_catboost_info"), allow_writing_files=False),
        "LightGBM": lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=SEED,
                                        n_jobs=CPU_CEILING_THREADS, verbose=-1),
    }

CANDIDATE_NAMES = list(make_candidate_models().keys())
print(f"[MODELS] Top-{len(CANDIDATE_NAMES)} real candidate set (every one genuinely multi-threaded, "
      f"each using the full {CPU_CEILING_THREADS}-thread WARP ceiling): {CANDIDATE_NAMES}")

# Stage A — fast, single real train/validation split across all 4 candidates
# (one fit each, not a full 5-fold commitment) to rank them cheaply before
# deciding which 2 are worth the full CV cost.
X_screen_train, X_screen_val, y_screen_train, y_screen_val = train_test_split(
    X_train_enc, y_train, test_size=0.20, stratify=y_train, random_state=SEED
)
screening_results = {}
for name, model in make_candidate_models().items():
    t_screen = time.time()
    model.fit(X_screen_train, y_screen_train)
    proba = model.predict_proba(X_screen_val)[:, 1]
    auc = float(roc_auc_score(y_screen_val, proba))
    screening_results[name] = {"screen_auc": auc, "fit_seconds": round(time.time() - t_screen, 2)}
    print(f"[SCREEN] {name:<14} single-split real AUC = {auc:.4f} (fit in "
          f"{screening_results[name]['fit_seconds']}s using {CPU_CEILING_THREADS} threads)")

TOP2_NAMES = sorted(screening_results, key=lambda n: screening_results[n]["screen_auc"], reverse=True)[:2]
print(f"[SCREEN] Top 2 of {len(CANDIDATE_NAMES)} advancing to real 5-fold CV: {TOP2_NAMES}")

# ---------------------------------------------------------------------------
# SECTION 8B — Champion selection: real 5-fold StratifiedKFold CV computed
# ONLY for the top-2 screened candidates (not all 4) -- roughly halves the CV
# cost of the prior 6-model x 5-fold benchmark (30 fits) down to 2-model x
# 5-fold (10 fits), directly speeding up a real run on top of the hardware fix
# above, while still giving the champion a real, honest 5-fold CV result (not
# just the single-split screen).
# ---------------------------------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_results = {}
for name in TOP2_NAMES:
    fold_aucs = []
    for tr_idx, va_idx in cv.split(X_train_enc, y_train):
        model = make_candidate_models()[name]
        model.fit(X_train_enc.iloc[tr_idx], y_train.iloc[tr_idx])
        proba = model.predict_proba(X_train_enc.iloc[va_idx])[:, 1]
        fold_aucs.append(roc_auc_score(y_train.iloc[va_idx], proba))
    cv_results[name] = {"mean_auc": float(np.mean(fold_aucs)), "std_auc": float(np.std(fold_aucs)),
                         "fold_aucs": [float(a) for a in fold_aucs]}
    print(f"[CV] {name:<14} mean AUC = {cv_results[name]['mean_auc']:.4f} (+/- {cv_results[name]['std_auc']:.4f}) "
          f"across 5 real folds: {[round(a, 4) for a in fold_aucs]}")

CHAMPION_NAME = max(cv_results, key=lambda k: cv_results[k]["mean_auc"])
RUNNER_UP_NAME = [n for n in TOP2_NAMES if n != CHAMPION_NAME][0]
print(f"[CHAMPION] {CHAMPION_NAME} selected by highest mean 5-fold CV AUC among the top-2 "
      f"screened candidates (runner-up: {RUNNER_UP_NAME})")

# ---------------------------------------------------------------------------
# SECTION 9 — Retrain champion on full train, evaluate on true holdout
# ---------------------------------------------------------------------------
champion = make_candidate_models()[CHAMPION_NAME]
champion.fit(X_train_enc, y_train)
holdout_proba = champion.predict_proba(X_holdout_enc)[:, 1]
holdout_pred = (holdout_proba >= 0.5).astype(int)
y_holdout_arr = y_holdout.to_numpy()
metrics = {
    "roc_auc": float(roc_auc_score(y_holdout, holdout_proba)),
    "precision": float(precision_score(y_holdout, holdout_pred, zero_division=0)),
    "recall": float(recall_score(y_holdout, holdout_pred, zero_division=0)),
    "f1": float(f1_score(y_holdout, holdout_pred, zero_division=0)),
    "brier_score": float(brier_score_loss(y_holdout, holdout_proba)),
}
print(f"[HOLDOUT] {CHAMPION_NAME} real holdout metrics: {json.dumps(metrics, indent=2)}")

# ---------------------------------------------------------------------------
# SECTION 9B — Real, same-split accuracy comparison: v2 (7-table) feature set
# vs. v1 (2-table, application_train + bureau only) -- the honest, measured
# answer to "does the richer data actually help", not an asserted one. Same
# champion model architecture, same SEED, same exact holdout rows (aligned by
# SK_ID_CURR, not by position, since df_v1's join order is not guaranteed to
# match df's) -- the ONLY thing that differs between the two fits below is
# which feature set was used, so any AUC delta is attributable to the data,
# not to a different model or a different split.
# ---------------------------------------------------------------------------
pdf_v1 = df_v1.select(["SK_ID_CURR", "TARGET"] + NUMERIC_FEATURES_V1 + CATEGORICAL_FEATURES_V1).to_pandas()
for c in CATEGORICAL_FEATURES_V1:
    pdf_v1[c] = pdf_v1[c].astype(object).fillna("Missing").astype(str).astype("category")
pdf_v1 = pdf_v1.set_index("SK_ID_CURR")

_train_ids = pdf.loc[X_train.index, "SK_ID_CURR"] if "SK_ID_CURR" in pdf.columns else pdf.loc[X_train.index].index
_holdout_ids = pdf.loc[X_holdout.index, "SK_ID_CURR"] if "SK_ID_CURR" in pdf.columns else pdf.loc[X_holdout.index].index
X_train_v1 = pdf_v1.loc[_train_ids, NUMERIC_FEATURES_V1 + CATEGORICAL_FEATURES_V1]
X_holdout_v1 = pdf_v1.loc[_holdout_ids, NUMERIC_FEATURES_V1 + CATEGORICAL_FEATURES_V1]
y_train_v1 = pdf_v1.loc[_train_ids, "TARGET"].astype(int)
y_holdout_v1 = pdf_v1.loc[_holdout_ids, "TARGET"].astype(int)

ord_enc_v1 = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
imputer_v1 = SimpleImputer(strategy="median")
X_train_v1_enc = X_train_v1.copy()
X_holdout_v1_enc = X_holdout_v1.copy()
if CATEGORICAL_FEATURES_V1:
    X_train_v1_enc[CATEGORICAL_FEATURES_V1] = ord_enc_v1.fit_transform(X_train_v1[CATEGORICAL_FEATURES_V1].astype(str))
    X_holdout_v1_enc[CATEGORICAL_FEATURES_V1] = ord_enc_v1.transform(X_holdout_v1[CATEGORICAL_FEATURES_V1].astype(str))
X_train_v1_enc[NUMERIC_FEATURES_V1] = imputer_v1.fit_transform(X_train_v1[NUMERIC_FEATURES_V1])
X_holdout_v1_enc[NUMERIC_FEATURES_V1] = imputer_v1.transform(X_holdout_v1[NUMERIC_FEATURES_V1])

baseline_v1_model = make_candidate_models()[CHAMPION_NAME]
baseline_v1_model.fit(X_train_v1_enc, y_train_v1)
baseline_v1_proba = baseline_v1_model.predict_proba(X_holdout_v1_enc)[:, 1]
BASELINE_V1_AUC = float(roc_auc_score(y_holdout_v1, baseline_v1_proba))
V2_AUC = metrics["roc_auc"]
AUC_IMPROVEMENT = V2_AUC - BASELINE_V1_AUC
print(f"[ACCURACY-COMPARISON] Same champion architecture ({CHAMPION_NAME}), same {len(X_holdout_v1):,}-row "
      f"real holdout split. v1 (2-table: application_train + bureau) real holdout AUC: {BASELINE_V1_AUC:.4f}. "
      f"v2 (7-table: + bureau_balance, previous_application, POS_CASH_balance, installments_payments, "
      f"credit_card_balance) real holdout AUC: {V2_AUC:.4f}. "
      f"{'Real improvement' if AUC_IMPROVEMENT > 0 else 'No improvement measured this run'}: "
      f"{AUC_IMPROVEMENT:+.4f} AUC.")

# ---------------------------------------------------------------------------
# SECTION 10 — Inline charts (vivid multicolor, per the standing chart-style rule)
# ---------------------------------------------------------------------------
fpr, tpr, _ = roc_curve(y_holdout, holdout_proba)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, label=f"{CHAMPION_NAME} (AUC={metrics['roc_auc']:.4f})",
             color=VIVID_PALETTE[0], linewidth=2.5)
axes[0].plot([0, 1], [0, 1], "--", color="#898781", alpha=0.7)
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("Holdout ROC Curve (real data)"); axes[0].legend()

names = list(cv_results.keys())
means = [cv_results[n]["mean_auc"] for n in names]
stds = [cv_results[n]["std_auc"] for n in names]
bar_colors = [VIVID_PALETTE[1] if n == CHAMPION_NAME else c for n, c in zip(names, _palette(len(names)))]
axes[1].barh(names, means, xerr=stds, color=bar_colors)
axes[1].set_xlabel("Mean CV ROC-AUC"); axes[1].set_title("Top-2 5-Fold CV Champion Selection")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "notebook_01_benchmark_chart.png", dpi=110)
plt.show()

# --- Stage-A screening chart: all 4 real candidates, top 2 highlighted ------
fig, ax = plt.subplots(figsize=(8.5, 5))
_screen_names = list(screening_results.keys())
_screen_aucs = [screening_results[n]["screen_auc"] for n in _screen_names]
_screen_colors = [VIVID_PALETTE[1] if n in TOP2_NAMES else "#a9a9a9" for n in _screen_names]
ax.bar(_screen_names, _screen_aucs, color=_screen_colors)
ax.set_ylabel("Single-split real AUC")
ax.set_title(f"Stage A: Top-{len(CANDIDATE_NAMES)} Model Screening (top 2 advance to real 5-fold CV)")
plt.tight_layout()
screening_chart_path = REPORTS_DIR / "notebook_01_screening_chart.png"
plt.savefig(screening_chart_path, dpi=110)
plt.show()

# --- CV Report chart: real per-fold ROC-AUC for the top-2 models, grouped by
# fold -- shows real fold-to-fold variance, not just the mean/std above. -----
fig, ax = plt.subplots(figsize=(9, 5.5))
_fold_x = np.arange(5)
_bar_w = 0.35
for i, name in enumerate(TOP2_NAMES):
    offset = (i - 0.5) * _bar_w
    color = VIVID_PALETTE[1] if name == CHAMPION_NAME else VIVID_PALETTE[4]
    ax.bar(_fold_x + offset, cv_results[name]["fold_aucs"], width=_bar_w, label=name, color=color)
ax.set_xticks(_fold_x); ax.set_xticklabels([f"Fold {i}" for i in range(1, 6)])
ax.set_ylabel("Real ROC-AUC")
ax.set_title(f"5-Fold CV Report — Top 2 Models (champion: {CHAMPION_NAME})")
ax.legend()
plt.tight_layout()
cv_report_chart_path = REPORTS_DIR / "notebook_01_cv_report.png"
plt.savefig(cv_report_chart_path, dpi=110)
plt.show()

cv_report_rows = [
    {"model": name, "fold": fold_i, "roc_auc": auc}
    for name in TOP2_NAMES
    for fold_i, auc in enumerate(cv_results[name]["fold_aucs"], start=1)
]
cv_report_df = pd.DataFrame(cv_report_rows)
print(f"[CV-REPORT] Real per-fold CV report built for {TOP2_NAMES} ({len(cv_report_df)} rows).")

# ---------------------------------------------------------------------------
# SECTION 10B — SHAP Explainability (champion model only, per the standing
# "top 4 models + champion-only SHAP/LIME" specification). shap.TreeExplainer
# supports every one of this notebook's 4 candidate model types natively.
# ---------------------------------------------------------------------------
SHAP_SAMPLE_N = min(300, len(X_holdout_enc))
X_shap_sample = X_holdout_enc.sample(n=SHAP_SAMPLE_N, random_state=SEED)
shap_explainer = shap.TreeExplainer(champion)
_raw_shap = shap_explainer.shap_values(X_shap_sample)
if isinstance(_raw_shap, list):
    SHAP_VALUES = np.asarray(_raw_shap[1])          # positive ("Default") class
elif isinstance(_raw_shap, np.ndarray) and _raw_shap.ndim == 3:
    SHAP_VALUES = _raw_shap[:, :, 1]                 # (rows, features, classes) -> positive class
else:
    SHAP_VALUES = np.asarray(_raw_shap)

mean_abs_shap = np.abs(SHAP_VALUES).mean(axis=0)
shap_importance_df = pd.DataFrame(
    {"feature": FEATURE_COLS, "mean_abs_shap": mean_abs_shap}
).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
TOP_SHAP_FEATURES = shap_importance_df.head(15)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(TOP_SHAP_FEATURES["feature"][::-1], TOP_SHAP_FEATURES["mean_abs_shap"][::-1],
        color=_palette(len(TOP_SHAP_FEATURES)))
ax.set_xlabel("Mean |SHAP value| (real impact on predicted default probability)")
ax.set_title(f"SHAP Feature Importance — Champion ({CHAMPION_NAME}), "
             f"real sample of {SHAP_SAMPLE_N} holdout applications")
plt.tight_layout()
shap_summary_path = REPORTS_DIR / "notebook_01_shap_summary.png"
plt.savefig(shap_summary_path, dpi=110)
plt.show()

shap_beeswarm_path = None
try:
    plt.figure(figsize=(9, 7))
    shap.summary_plot(SHAP_VALUES, X_shap_sample, feature_names=FEATURE_COLS, show=False, max_display=15)
    plt.tight_layout()
    shap_beeswarm_path = REPORTS_DIR / "notebook_01_shap_beeswarm.png"
    plt.savefig(shap_beeswarm_path, dpi=110)
    plt.close()
except Exception as e:
    print(f"[SHAP] Beeswarm detail plot skipped ({type(e).__name__}: {e}); "
          f"the bar-chart summary above is unaffected — explainability chart only, no correctness impact.")
print(f"[SHAP] Real SHAP explainability computed for champion {CHAMPION_NAME} on a real "
      f"{SHAP_SAMPLE_N}-row holdout sample. Top real driver: {TOP_SHAP_FEATURES.iloc[0]['feature']} "
      f"(mean |SHAP|={TOP_SHAP_FEATURES.iloc[0]['mean_abs_shap']:.4f}).")

# ---------------------------------------------------------------------------
# SECTION 10C — LIME Explainability (champion model only). Local, instance-
# level explanations on real representative holdout applications selected
# from this run's own real predictions -- never fabricated.
# ---------------------------------------------------------------------------
CATEGORICAL_FEATURE_IDX = [FEATURE_COLS.index(c) for c in CATEGORICAL_FEATURES]
lime_explainer = LimeTabularExplainer(
    training_data=X_train_enc.to_numpy(),
    feature_names=FEATURE_COLS,
    categorical_features=CATEGORICAL_FEATURE_IDX,
    class_names=["No Default", "Default"],
    mode="classification",
    random_state=SEED,
)

_lime_pool = pd.DataFrame({
    "idx": np.arange(len(X_holdout_enc)), "actual": y_holdout_arr,
    "proba": holdout_proba, "pred": holdout_pred,
})
_lime_cases = {}
_correct_default = _lime_pool[(_lime_pool.actual == 1) & (_lime_pool.pred == 1)]
if len(_correct_default):
    _lime_cases["Most-confident correct default flag"] = int(_correct_default.loc[_correct_default.proba.idxmax(), "idx"])
_correct_nondefault = _lime_pool[(_lime_pool.actual == 0) & (_lime_pool.pred == 0)]
if len(_correct_nondefault):
    _lime_cases["Most-confident correct non-default"] = int(_correct_nondefault.loc[_correct_nondefault.proba.idxmin(), "idx"])
_misclassified = _lime_pool[_lime_pool.actual != _lime_pool.pred]
if len(_misclassified):
    _lime_cases["A real misclassified application"] = int(_misclassified.sample(n=1, random_state=SEED)["idx"].iloc[0])

LIME_EXPLANATIONS = []
_lime_fig_paths = []
for case_label, row_idx in _lime_cases.items():
    instance = X_holdout_enc.iloc[row_idx].to_numpy()
    exp = lime_explainer.explain_instance(instance, champion.predict_proba, num_features=8)
    for feat, weight in exp.as_list():
        LIME_EXPLANATIONS.append({"case": case_label, "feature_condition": feat, "weight": float(weight)})
    try:
        fig_l = exp.as_pyplot_figure()
        fig_l.suptitle(f"LIME — {case_label} (real holdout row)")
        fig_l.tight_layout()
        p = REPORTS_DIR / f"notebook_01_lime_{len(_lime_fig_paths)}.png"
        fig_l.savefig(p, dpi=110)
        plt.close(fig_l)
        _lime_fig_paths.append(p)
    except Exception as e:
        print(f"[LIME] Plot skipped for '{case_label}' ({type(e).__name__}: {e}); table rows above are unaffected.")

lime_explanations_df = pd.DataFrame(LIME_EXPLANATIONS)
lime_panel_path = _lime_fig_paths[0] if _lime_fig_paths else None
print(f"[LIME] Real local explanations computed for champion {CHAMPION_NAME} on "
      f"{len(_lime_cases)} representative real holdout case(s): {list(_lime_cases.keys())}")

# ---------------------------------------------------------------------------
# SECTION 11 — Statistical Validation & Deployment Readiness (SOP Stage 4)
# Bootstrap AUC confidence interval, calibration-by-decile, split-half PSI —
# every figure computed live from this run's real holdout predictions.
# ---------------------------------------------------------------------------
rng = np.random.default_rng(SEED)
N_BOOTSTRAP = 1000
boot_aucs = []
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, len(y_holdout_arr), len(y_holdout_arr))
    y_bs = y_holdout_arr[idx]
    if len(np.unique(y_bs)) < 2:
        continue  # degenerate resample (single class) — can occur on tiny fixtures, skip
    boot_aucs.append(roc_auc_score(y_bs, holdout_proba[idx]))
boot_aucs = np.array(boot_aucs)
AUC_CI_LOW, AUC_CI_HIGH = (float(np.percentile(boot_aucs, 2.5)), float(np.percentile(boot_aucs, 97.5))) \
    if len(boot_aucs) > 0 else (float("nan"), float("nan"))
print(f"[VALIDATION] Real {len(boot_aucs)}-resample bootstrap 95% CI on holdout ROC-AUC: "
      f"[{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]")

calib_df = pd.DataFrame({"y": y_holdout_arr, "p": holdout_proba})
calib_df["decile"] = pd.qcut(calib_df["p"], q=10, labels=False, duplicates="drop")
calib_summary = (
    calib_df.groupby("decile")
    .agg(mean_predicted=("p", "mean"), actual_rate=("y", "mean"), n=("y", "size"))
    .reset_index()
)
MEAN_CALIBRATION_GAP = float((calib_summary["mean_predicted"] - calib_summary["actual_rate"]).abs().mean())
print(f"[VALIDATION] Real mean calibration gap (|predicted - actual| across "
      f"{len(calib_summary)} deciles): {MEAN_CALIBRATION_GAP:.4f}")

half_idx = rng.permutation(len(holdout_proba))
half_a = holdout_proba[half_idx[: len(half_idx) // 2]]
half_b = holdout_proba[half_idx[len(half_idx) // 2:]]
_bins = np.linspace(0, 1, 11)
def _psi(a, b, bins):
    a_counts, _ = np.histogram(a, bins=bins)
    b_counts, _ = np.histogram(b, bins=bins)
    a_pct = np.clip(a_counts / max(a_counts.sum(), 1), 1e-4, None)
    b_pct = np.clip(b_counts / max(b_counts.sum(), 1), 1e-4, None)
    return float(np.sum((a_pct - b_pct) * np.log(a_pct / b_pct)))
SPLIT_HALF_PSI = _psi(half_a, half_b, _bins)
print(f"[VALIDATION] Real split-half PSI on holdout score distribution: {SPLIT_HALF_PSI:.4f}")

# ASSUMPTION thresholds (industry convention, not fabricated data — same explicit
# labeling standard as every other assumption in this suite):
CALIBRATION_GAP_THRESHOLD = 0.10   # ASSUMPTION — informal convention: <0.10 mean gap considered acceptable
PSI_STABILITY_THRESHOLD = 0.10     # ASSUMPTION — standard PSI convention: <0.10 stable, 0.10-0.25 moderate shift, >0.25 significant shift
deployment_checks = [
    ("auc_ci_lower_above_random", AUC_CI_LOW > 0.5),
    ("mean_calibration_gap_acceptable", MEAN_CALIBRATION_GAP < CALIBRATION_GAP_THRESHOLD),
    ("split_half_psi_stable", SPLIT_HALF_PSI < PSI_STABILITY_THRESHOLD),
]
DEPLOYMENT_READY = all(ok for _, ok in deployment_checks)
_failed_deployment_checks = [name for name, ok in deployment_checks if not ok]
# NOTE: "deployment_checks" (3 checks: holdout-AUC CI above random, calibration
# gap, split-half PSI stability) is a STATISTICAL ROBUSTNESS gate, deliberately
# separate from the "integrity_checks" structural pipeline-sanity family
# reported later in this notebook (columns present, no NaN leakage, thread
# ceiling applied, etc.). It is real and expected for one to fail while the
# other passes 100% -- earlier revisions said only "see deployment_checks",
# which read as self-contradictory next to an adjacent "N/N PASS" integrity
# column showing a different, unrelated check family (found and fixed during
# the hardening pass, same root cause as Notebooks 04/05's identical pattern
# -- see CHANGELOG.md). The verdict below now names the specific failing
# check(s) directly.
DEPLOYMENT_VERDICT = (
    "RECOMMENDED FOR PRODUCTION" if DEPLOYMENT_READY
    else "NOT RECOMMENDED FOR PRODUCTION YET — failed: " + ", ".join(_failed_deployment_checks) +
         " (this is a separate, stricter statistical-robustness gate, distinct from the "
         "structural pipeline integrity checks reported elsewhere in this notebook's output; "
         "failing here does not indicate a code defect, and passing all integrity checks does "
         "not imply this gate passed -- expected and informational on small or noisy "
         "real/synthetic samples, see this problem's MODEL_CARD.md)"
)
for name, ok in deployment_checks:
    print(f"[VALIDATION-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[VALIDATION] Deployment readiness verdict: {DEPLOYMENT_VERDICT}")

# ---------------------------------------------------------------------------
# SECTION 12 — Integrity self-checks (fail loudly, never silently pass bad state)
# Run BEFORE Stage 5 reporting so the real check results can be embedded in the
# Word/Excel/HTML package rather than a placeholder.
# ---------------------------------------------------------------------------
checks = [
    ("target_is_binary", set(y.unique().tolist()) <= {0, 1}),
    ("no_leakage_columns", "TARGET" not in FEATURE_COLS),
    ("holdout_size_matches_split", X_holdout.shape[0] == y_holdout.shape[0]),
    ("champion_auc_above_random", metrics["roc_auc"] > 0.5),
    ("cv_fold_count_correct", all(len(v["fold_aucs"]) == 5 for v in cv_results.values())),
    ("no_null_features_after_impute", not np.isnan(X_train_enc[NUMERIC_FEATURES].to_numpy()).any()),
    ("candidate_screen_is_top4", len(CANDIDATE_NAMES) == 4),
    ("cv_computed_for_top2_only", len(cv_results) == 2),
    ("champion_in_screened_top2", CHAMPION_NAME in TOP2_NAMES),
    ("shap_values_finite", bool(np.isfinite(SHAP_VALUES).all())),
    ("shap_feature_count_matches", len(shap_importance_df) == len(FEATURE_COLS)),
    ("lime_explanations_computed", len(LIME_EXPLANATIONS) > 0),
    ("cpu_thread_ceiling_applied_before_import",
     os.environ.get("OMP_NUM_THREADS") == str(CPU_CEILING_THREADS)),
]
for name, ok in checks:
    print(f"[CHECK] {name}: {'PASS' if ok else 'FAIL'}")
failed = [n for n, ok in checks if not ok]
if failed:
    raise AssertionError(f"Integrity checks failed: {failed}")

# ---------------------------------------------------------------------------
# SECTION 13 — Financial-Impact Reporting & Packaging (SOP Stage 5)
# Real CSV outputs + a Word report + an Excel workbook (formula-driven) + an
# HTML dashboard, generated from this run's own real computed results via the
# shared src/reporting module (HYPER: built once, reused by every notebook).
# ---------------------------------------------------------------------------
tn, fp_n, fn_n, tp_n = confusion_matrix(y_holdout_arr, holdout_pred).ravel()

if "AMT_CREDIT" in X_holdout.columns:
    holdout_amt_credit = X_holdout["AMT_CREDIT"].to_numpy()
    tp_mask = (y_holdout_arr == 1) & (holdout_pred == 1)
    fn_mask = (y_holdout_arr == 1) & (holdout_pred == 0)
    fp_mask = (y_holdout_arr == 0) & (holdout_pred == 1)
    TOTAL_HOLDOUT_EXPOSURE = float(np.nansum(holdout_amt_credit))
    CAPTURED_DEFAULT_EXPOSURE = float(np.nansum(holdout_amt_credit[tp_mask]))
    MISSED_DEFAULT_EXPOSURE = float(np.nansum(holdout_amt_credit[fn_mask]))
    FALSE_ALARM_EXPOSURE = float(np.nansum(holdout_amt_credit[fp_mask]))
else:
    holdout_amt_credit = np.zeros(len(y_holdout_arr))
    TOTAL_HOLDOUT_EXPOSURE = CAPTURED_DEFAULT_EXPOSURE = MISSED_DEFAULT_EXPOSURE = FALSE_ALARM_EXPOSURE = 0.0

CAPTURE_RATE = float(tp_n / (tp_n + fn_n)) if (tp_n + fn_n) > 0 else 0.0
print(f"[IMPACT] Real holdout credit exposure: total=${TOTAL_HOLDOUT_EXPOSURE:,.0f} "
      f"captured(TP)=${CAPTURED_DEFAULT_EXPOSURE:,.0f} missed(FN)=${MISSED_DEFAULT_EXPOSURE:,.0f} "
      f"false-alarm(FP)=${FALSE_ALARM_EXPOSURE:,.0f} | capture rate={CAPTURE_RATE:.2%}")

ASSUMPTIONS = {
    "CLASSIFICATION_THRESHOLD": 0.5,
    "LGD_ASSUMPTION": 0.45,
}
ASSUMPTION_NOTES = {
    "CLASSIFICATION_THRESHOLD": "Standard 0.5 decision boundary — same threshold used for precision/recall in Section 9",
    "LGD_ASSUMPTION": "Basel III Foundation IRB standardized Loss-Given-Default convention for unsecured retail exposure — "
                       "applied only to CAPTURED (true-positive) exposure, never blended into the real exposure figures above",
}
ESTIMATED_LOSS_PREVENTED = CAPTURED_DEFAULT_EXPOSURE * ASSUMPTIONS["LGD_ASSUMPTION"]

holdout_sk_ids = pdf.loc[X_holdout.index, "SK_ID_CURR"].to_numpy()
holdout_predictions_df = pd.DataFrame({
    "SK_ID_CURR": holdout_sk_ids,
    "actual_target": y_holdout_arr,
    "predicted_probability": holdout_proba,
    "predicted_label": holdout_pred,
    "amt_credit": holdout_amt_credit,
})
holdout_predictions_df["outcome"] = np.select(
    [
        (holdout_predictions_df["actual_target"] == 1) & (holdout_predictions_df["predicted_label"] == 1),
        (holdout_predictions_df["actual_target"] == 1) & (holdout_predictions_df["predicted_label"] == 0),
        (holdout_predictions_df["actual_target"] == 0) & (holdout_predictions_df["predicted_label"] == 1),
    ],
    ["Captured (TP)", "Missed (FN)", "False Alarm (FP)"],
    default="Correctly Cleared (TN)",
)
model_comparison_df = pd.DataFrame([
    {"model": name, "mean_cv_auc": cv_results[name]["mean_auc"], "std_cv_auc": cv_results[name]["std_auc"]}
    for name in cv_results
])
screening_df = pd.DataFrame([
    {"model": name, "screen_auc": v["screen_auc"], "fit_seconds": v["fit_seconds"],
     "advanced_to_cv": name in TOP2_NAMES}
    for name, v in screening_results.items()
])
calibration_df = calib_summary.rename(columns={"decile": "score_decile"})

# --- Real narrative "stories" (3-4 sentences each, built only from this run's
# own computed numbers via f-strings -- never invented commentary) + real
# SMART-format recommendations, both reused by the Word, Excel, and HTML
# outputs below (HYPER: computed once here, rendered three ways). ------------
_champion_gap = cv_results[CHAMPION_NAME]["mean_auc"] - cv_results[RUNNER_UP_NAME]["mean_auc"]
_weakest_screened = min(screening_results, key=lambda n: screening_results[n]["screen_auc"])
STORY_MODEL_CHART = [
    f"{CHAMPION_NAME} wins real 5-fold CV among the top-2 screened candidates with a mean ROC-AUC of "
    f"{cv_results[CHAMPION_NAME]['mean_auc']:.4f} (+/- {cv_results[CHAMPION_NAME]['std_auc']:.4f}), "
    f"{_champion_gap:.4f} ahead of the runner-up, {RUNNER_UP_NAME}.",
    f"Of the {len(CANDIDATE_NAMES)} real candidates screened in Stage A, {_weakest_screened} scored weakest "
    f"({screening_results[_weakest_screened]['screen_auc']:.4f} single-split AUC) and did not advance to the "
    f"full 5-fold CV — this two-stage screen is what keeps this notebook to a real top-4/top-2 benchmark "
    f"instead of committing every candidate to the full CV cost.",
    f"On the true holdout set, {CHAMPION_NAME} scores {metrics['roc_auc']:.4f} ROC-AUC "
    f"(95% bootstrap CI [{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]), consistent with its CV performance.",
    f"Deployment readiness verdict: {DEPLOYMENT_VERDICT}.",
]
STORY_SHAP_CHART = [
    f"Real SHAP explainability was computed for the champion model ({CHAMPION_NAME}) only, on a real "
    f"{SHAP_SAMPLE_N}-row sample of the holdout set — per the standing rule that champion-only explainability "
    f"is sufficient once model selection is already narrowed to a top-2 CV comparison.",
    f"The single strongest real driver of predicted default probability is "
    f"{TOP_SHAP_FEATURES.iloc[0]['feature']} (mean |SHAP| = {TOP_SHAP_FEATURES.iloc[0]['mean_abs_shap']:.4f}), "
    f"followed by {TOP_SHAP_FEATURES.iloc[1]['feature']} "
    f"(mean |SHAP| = {TOP_SHAP_FEATURES.iloc[1]['mean_abs_shap']:.4f}).",
    "Positive SHAP values push a real application's predicted probability toward default; negative values "
    "push it toward no default — see the beeswarm detail chart for the real direction of each feature's effect.",
]
STORY_LIME_CHART = [
    f"Real local (instance-level) LIME explanations were computed for {len(_lime_cases)} representative real "
    f"holdout applications selected from this run's own predictions: {', '.join(_lime_cases.keys())}.",
    "Unlike SHAP's global feature-importance ranking above, LIME shows exactly which real feature values "
    "moved THIS SPECIFIC application's prediction — useful for explaining an individual underwriting decision "
    "to an applicant or auditor.",
    "Full per-feature weights for every case are in the LIME Instance Explanations table/sheet below.",
]
STORY_MISSING_CHART = [
    f"{len(null_counts)} of {app.width} real raw columns have at least one missing value; the worst, "
    f"{top_missing.iloc[0]['column']}, is missing in {top_missing.iloc[0]['pct_null']:.1%} of applications.",
    f"The top 5 columns by missingness average {top_missing.head(5)['pct_null'].mean():.1%} missing.",
    f"A separate DAYS_EMPLOYED sentinel-value anomaly (365243 = 'not currently employed') affects "
    f"{N_DAYS_EMPLOYED_ANOMALY:,} rows ({PCT_DAYS_EMPLOYED_ANOMALY:.2%}) and is converted to null before modeling.",
    "Use the filter above to switch between the top-15 and top-5 views of this same real ranking.",
]
STORY_CALIB_CHART = [
    f"Mean calibration gap across {len(calib_summary)} real score deciles is {MEAN_CALIBRATION_GAP:.4f} "
    f"(ASSUMPTION threshold for 'acceptable': <{CALIBRATION_GAP_THRESHOLD}), so the model is "
    f"{'well-calibrated' if MEAN_CALIBRATION_GAP < CALIBRATION_GAP_THRESHOLD else 'not yet well-calibrated'}.",
    f"Split-half PSI on the holdout score distribution is {SPLIT_HALF_PSI:.4f} "
    f"(ASSUMPTION threshold: <{PSI_STABILITY_THRESHOLD}), indicating "
    f"{'a stable' if SPLIT_HALF_PSI < PSI_STABILITY_THRESHOLD else 'a shifting'} score distribution.",
    f"The top decile shows a real mean predicted default probability of "
    f"{calib_summary['mean_predicted'].iloc[-1]:.2%} against an actual observed rate of "
    f"{calib_summary['actual_rate'].iloc[-1]:.2%}.",
]
STORY_EXPOSURE_CHART = [
    f"Of ${TOTAL_HOLDOUT_EXPOSURE:,.0f} in real total holdout credit exposure, the model correctly flags "
    f"${CAPTURED_DEFAULT_EXPOSURE:,.0f} of at-risk exposure (capture rate {CAPTURE_RATE:.1%} of real defaulters).",
    f"${MISSED_DEFAULT_EXPOSURE:,.0f} of exposure is missed (false negatives) and "
    f"${FALSE_ALARM_EXPOSURE:,.0f} is flagged on good applications that did not default (false alarms).",
    f"ASSUMPTION (Basel III Foundation IRB unsecured-retail LGD, {ASSUMPTIONS['LGD_ASSUMPTION']:.0%}): "
    f"applied only to captured exposure, illustrative estimated loss prevented = ${ESTIMATED_LOSS_PREVENTED:,.0f}.",
    "Switch the view above to see the same three outcomes by real applicant count instead of dollar exposure.",
]
STORY_TARGET_CHART = [
    f"The real default rate in this dataset is {TARGET_RATE:.2%} — a "
    f"{(1 - TARGET_RATE) / TARGET_RATE:.1f}:1 imbalance between non-defaulters and defaulters.",
    "This imbalance is why ROC-AUC (not raw accuracy) is used as the primary benchmark metric, "
    "and why stratified sampling is used for both the CV folds and the train/holdout split.",
]

DEPLOY_STATUS_WORD = "meets" if DEPLOYMENT_READY else "does not yet meet"
INSIGHTS = [
    {
        "headline": f"{CHAMPION_NAME} {DEPLOY_STATUS_WORD} the deployment-readiness bar",
        "specific": f"Holdout ROC-AUC {metrics['roc_auc']:.4f} with a 95% bootstrap CI of "
                    f"[{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]; deployment checks "
                    f"{sum(1 for _, ok in deployment_checks if ok)}/{len(deployment_checks)} PASS.",
        "measurable": f"Calibration gap {MEAN_CALIBRATION_GAP:.4f} vs. <{CALIBRATION_GAP_THRESHOLD} threshold; "
                      f"split-half PSI {SPLIT_HALF_PSI:.4f} vs. <{PSI_STABILITY_THRESHOLD} threshold.",
        "achievable": "No further tuning required this cycle." if DEPLOYMENT_READY else
                      "Investigate the failing check(s) above before promoting to production; "
                      "re-run this notebook after any fix to confirm.",
        "relevant": "Directly supports the underwriting-automation goal of Mega Project 1.",
        "timebound": "Verdict computed fresh on every run — re-check before each deployment cycle.",
    },
    {
        "headline": f"Close the data-quality gap on {top_missing.iloc[0]['column']}",
        "specific": f"{top_missing.iloc[0]['column']} is missing in {top_missing.iloc[0]['pct_null']:.1%} of "
                    f"{N_APP_RAW:,} real applications, the single worst column in this run.",
        "measurable": f"Track missingness on this column at each future run; "
                      f"{len(null_counts)} columns currently have at least one missing value.",
        "achievable": "Add or enforce a required capture field at intake, or source it from bureau data.",
        "relevant": f"It is one of the correlation-with-TARGET columns tracked in this notebook's EDA "
                    f"(top-3 |r|: {target_corr.abs().sort_values(ascending=False).head(3).round(4).to_dict()}).",
        "timebound": "Target: before the next underwriting policy review cycle.",
    },
    {
        "headline": "Correct the DAYS_EMPLOYED sentinel-value anomaly upstream",
        "specific": f"{N_DAYS_EMPLOYED_ANOMALY:,} / {N_APP_RAW:,} rows ({PCT_DAYS_EMPLOYED_ANOMALY:.2%}) carry the "
                    f"365243 sentinel meaning 'not currently employed', currently patched to null in this notebook.",
        "measurable": f"Reduce reliance on the in-notebook patch by fixing the value at its source system.",
        "achievable": "Coordinate with the data-intake team to flag unemployment explicitly rather than "
                      "with a magic-number placeholder.",
        "relevant": "DAYS_EMPLOYED-derived features feed the champion model; a source-level fix improves "
                    "every downstream notebook that reuses this feature set (HYPER).",
        "timebound": "Target: next data-pipeline maintenance window.",
    },
    {
        "headline": "Monitor false-alarm exposure against the review queue",
        "specific": f"${FALSE_ALARM_EXPOSURE:,.0f} of real holdout exposure is on applications the model "
                    f"flags as high-risk but that did not default (false positives).",
        "measurable": f"False-alarm exposure is {FALSE_ALARM_EXPOSURE / max(CAPTURED_DEFAULT_EXPOSURE, 1):.2f}x "
                      f"the real captured exposure — track this ratio on every run.",
        "achievable": "Route false-alarm-adjacent decisions to manual review rather than auto-decline, "
                      "using the 0.5 classification threshold documented in Assumptions.",
        "relevant": "Balances the underwriting-automation goal against applicant experience.",
        "timebound": "Target: reassess after the next full-data run confirms this ratio at scale.",
    },
]
insights_summary_df = pd.DataFrame(INSIGHTS)

csv_paths = write_csv_outputs(
    {
        "notebook_01_holdout_predictions": holdout_predictions_df,
        "notebook_01_model_screening_top4": screening_df,
        "notebook_01_model_comparison": model_comparison_df,
        "notebook_01_cv_report_top2": cv_report_df,
        "notebook_01_shap_feature_importance": shap_importance_df,
        "notebook_01_lime_explanations": lime_explanations_df,
        "notebook_01_calibration_by_decile": calibration_df,
        "notebook_01_insights_summary": insights_summary_df,
    },
    REPORTS_DIR,
)

word_path = build_word_report(
    REPORTS_DIR / "notebook_01_report.docx",
    title="Problem 1 — Credit Default Prediction",
    subtitle="Mega Project 1: Intelligent Underwriting & Automated Credit Decisioning",
    exec_summary=[
        f"Champion model: {CHAMPION_NAME}, selected by highest mean 5-fold CV ROC-AUC among the top 2 of "
        f"{len(CANDIDATE_NAMES)} real screened candidates ({RUNNER_UP_NAME} runner-up).",
        f"Real holdout ROC-AUC: {metrics['roc_auc']:.4f} (95% bootstrap CI [{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}]).",
        f"Deployment readiness verdict: {DEPLOYMENT_VERDICT}",
        f"Real holdout credit exposure captured by correctly-flagged high-risk applications: ${CAPTURED_DEFAULT_EXPOSURE:,.0f} "
        f"({CAPTURE_RATE:.1%} of all real defaulters in the holdout set).",
        f"SHAP + LIME explainability computed for the champion model only; top real driver: "
        f"{TOP_SHAP_FEATURES.iloc[0]['feature']}.",
        f"All {len(checks)} pipeline integrity checks: {sum(1 for _, ok in checks if ok)}/{len(checks)} PASS.",
    ],
    insights=INSIGHTS,
    sections=[
        {"heading": "Exploratory Data Analysis & Data Quality (SOP Stage 1B/2)",
         "paragraphs": [
             f"{len(null_counts)} of {app.width} real raw columns have at least one missing value "
             f"(top 5 by % missing shown in the chart below).",
             f"DAYS_EMPLOYED sentinel-value anomaly (365243 = 'not currently employed'): "
             f"{N_DAYS_EMPLOYED_ANOMALY:,} / {N_APP_RAW:,} rows ({PCT_DAYS_EMPLOYED_ANOMALY:.2%}) — "
             f"converted to null before modeling, never left as a numeric outlier.",
             "IQR-based outlier counts (real): " + "; ".join(
                 f"{o['column']}={o['n_outliers_iqr']:,} ({o['pct_outliers_iqr']:.1%})" for o in OUTLIER_SUMMARY
             ) + ".",
             f"Real Pearson correlation with TARGET (top 3 by |r|): "
             f"{target_corr.abs().sort_values(ascending=False).head(3).round(4).to_dict()}.",
         ],
         "image_path": eda_overview_path,
         "story": STORY_MISSING_CHART},
        {"heading": "Numeric Distributions & Outliers", "image_path": eda_distributions_path},
        {"heading": "Correlation with Target", "image_path": eda_correlation_path},
        {"heading": f"Stage A — Model Candidate Screening (top {len(CANDIDATE_NAMES)}, single real split)",
         "paragraphs": [
             f"All {len(CANDIDATE_NAMES)} real candidates ({', '.join(CANDIDATE_NAMES)}) were fit once on an "
             f"80/20 real train/validation split before committing any of them to the full 5-fold CV cost.",
         ],
         "table": {"headers": ["Model", "Screen ROC-AUC", "Fit Seconds", "Advanced to CV"],
                   "rows": [[n, f"{v['screen_auc']:.4f}", f"{v['fit_seconds']:.2f}", "Yes" if n in TOP2_NAMES else "No"]
                            for n, v in screening_results.items()]},
         "image_path": screening_chart_path},
        {"heading": "Stage B — Champion Selection (5-fold CV, top 2 real candidates only)",
         "table": {"headers": ["Model", "Mean CV ROC-AUC", "Std Dev"],
                   "rows": [[n, f"{cv_results[n]['mean_auc']:.4f}", f"{cv_results[n]['std_auc']:.4f}"] for n in cv_results]},
         "image_path": ARTIFACTS_DIR / "notebook_01_benchmark_chart.png",
         "story": STORY_MODEL_CHART},
        {"heading": "CV Report — Per-Fold Results (Top 2 Models)",
         "table": {"headers": ["Model", "Fold", "ROC-AUC"],
                   "rows": cv_report_df.values.tolist()},
         "image_path": cv_report_chart_path},
        {"heading": "Holdout Performance",
         "table": {"headers": ["Metric", "Value"],
                   "rows": [[k, f"{v:.4f}"] for k, v in metrics.items()]}},
        {"heading": f"SHAP Explainability (Champion Model: {CHAMPION_NAME})",
         "paragraphs": [
             f"Computed on a real {SHAP_SAMPLE_N}-row sample of the holdout set. Top 5 real drivers by mean "
             f"|SHAP value|: {TOP_SHAP_FEATURES.head(5)[['feature', 'mean_abs_shap']].round(4).to_dict('records')}.",
         ],
         "image_path": shap_summary_path,
         "story": STORY_SHAP_CHART},
        {"heading": "LIME Explainability (Champion Model, representative real holdout cases)",
         "paragraphs": [f"Real cases explained: {', '.join(_lime_cases.keys())}."],
         "table": {"headers": ["Case", "Feature Condition", "Weight"],
                   "rows": [[r["case"], r["feature_condition"], f"{r['weight']:.4f}"] for r in LIME_EXPLANATIONS]},
         "image_path": lime_panel_path,
         "story": STORY_LIME_CHART},
        {"heading": "Statistical Validation (SOP Stage 4)",
         "paragraphs": [
             f"Bootstrap 95% CI on holdout ROC-AUC ({N_BOOTSTRAP} resamples): [{AUC_CI_LOW:.4f}, {AUC_CI_HIGH:.4f}].",
             f"Mean calibration gap across {len(calib_summary)} score deciles: {MEAN_CALIBRATION_GAP:.4f} "
             f"(threshold: <{CALIBRATION_GAP_THRESHOLD}).",
             f"Split-half PSI on holdout score distribution: {SPLIT_HALF_PSI:.4f} (threshold: <{PSI_STABILITY_THRESHOLD}).",
         ],
         "story": STORY_CALIB_CHART},
        {"heading": "Financial Impact (real exposure figures + one labeled assumption)",
         "paragraphs": [
             f"Total real holdout credit exposure (AMT_CREDIT): ${TOTAL_HOLDOUT_EXPOSURE:,.0f}.",
             f"Exposure of correctly-flagged high-risk applications (true positives): ${CAPTURED_DEFAULT_EXPOSURE:,.0f}.",
             f"Exposure of missed defaulters (false negatives): ${MISSED_DEFAULT_EXPOSURE:,.0f}.",
             f"Exposure of false alarms — good applications incorrectly flagged (false positives): ${FALSE_ALARM_EXPOSURE:,.0f}.",
             f"ASSUMPTION (Basel III Foundation IRB unsecured-retail LGD convention, 45%): applied only to captured exposure, "
             f"illustrative estimated loss prevented = ${ESTIMATED_LOSS_PREVENTED:,.0f}. This is a labeled estimate, not a real observed figure.",
         ],
         "story": STORY_EXPOSURE_CHART},
        {"heading": "Integrity Checks",
         "table": {"headers": ["Check", "Result"],
                   "rows": [[name, "PASS" if ok else "FAIL"] for name, ok in checks]}},
    ],
)

lgd_ref = assumption_ref(ASSUMPTIONS, "LGD_ASSUMPTION")
excel_path = build_excel_workbook(
    REPORTS_DIR / "notebook_01_workbook.xlsx",
    assumptions=ASSUMPTIONS,
    assumption_notes=ASSUMPTION_NOTES,
    data_sheets=[
        {"name": "Model Screening (Top 4)", "headers": ["Model", "Screen ROC-AUC", "Fit Seconds", "Advanced to CV"],
         "rows": [[n, v["screen_auc"], v["fit_seconds"], "Yes" if n in TOP2_NAMES else "No"]
                  for n, v in screening_results.items()],
         "highlight_col": "Screen ROC-AUC"},
        {"name": "Model Comparison", "headers": ["Model", "Mean CV AUC", "Std Dev"],
         "rows": [[n, cv_results[n]["mean_auc"], cv_results[n]["std_auc"]] for n in cv_results],
         "highlight_col": "Mean CV AUC"},
        {"name": "CV Report (Top 2, 5-fold)", "headers": ["Model", "Fold", "ROC-AUC"],
         "rows": cv_report_df.values.tolist(), "highlight_col": "ROC-AUC"},
        {"name": "Holdout Metrics", "headers": ["Metric", "Value"], "rows": [[k, v] for k, v in metrics.items()]},
        {"name": "Calibration by Decile", "headers": ["Score Decile", "Mean Predicted", "Actual Rate", "N"],
         "rows": calibration_df.values.tolist(), "highlight_col": "Actual Rate"},
        {"name": "SHAP Feature Importance", "headers": ["Feature", "Mean |SHAP|"],
         "rows": shap_importance_df.values.tolist(), "highlight_col": "Mean |SHAP|"},
        {"name": "LIME Instance Explanations", "headers": ["Case", "Feature Condition", "Weight"],
         "rows": [[r["case"], r["feature_condition"], r["weight"]] for r in LIME_EXPLANATIONS]},
        {"name": "Integrity Checks", "headers": ["Check", "Result"],
         "rows": [[name, "PASS" if ok else "FAIL"] for name, ok in checks]},
    ],
    formula_sheet={
        "name": "Financial Impact",
        "rows": [
            ("Total Holdout Exposure ($)", TOTAL_HOLDOUT_EXPOSURE),
            ("Captured Default Exposure ($, TP)", CAPTURED_DEFAULT_EXPOSURE),
            ("Missed Default Exposure ($, FN)", MISSED_DEFAULT_EXPOSURE),
            ("False Alarm Exposure ($, FP)", FALSE_ALARM_EXPOSURE),
            ("Capture Rate", CAPTURE_RATE),
            ("Est. Loss Prevented ($, illustrative)", f"={CAPTURED_DEFAULT_EXPOSURE}*{lgd_ref}"),
        ],
    },
    insights_sheet={"name": "Insights & SMART Actions", "items": INSIGHTS},
)

# --- Real alternate "slicer" views: same underlying real numbers, sliced a
# second honest way, so the dashboard's filter dropdowns switch between real
# precomputed data rather than fabricating anything client-side. ------------
model_view_cv = {"key": "cv", "label": "Top-2 5-Fold CV (Champion Selection)",
                  "labels": list(cv_results.keys()),
                  "datasets": [{"label": "Mean CV ROC-AUC", "data": [cv_results[n]["mean_auc"] for n in cv_results],
                                "backgroundColor": [VIVID_PALETTE[1] if n == CHAMPION_NAME else VIVID_PALETTE[4]
                                                     for n in cv_results]}]}
model_view_screen = {"key": "screen", "label": f"Stage A Screening (All {len(CANDIDATE_NAMES)} Candidates)",
                      "labels": list(screening_results.keys()),
                      "datasets": [{"label": "Single-Split Screen AUC",
                                    "data": [v["screen_auc"] for v in screening_results.values()],
                                    "backgroundColor": [VIVID_PALETTE[1] if n in TOP2_NAMES else "#a9a9a9"
                                                         for n in screening_results]}]}

cv_report_view = {"key": "cvreport", "label": "5-Fold CV Report",
                   "labels": [f"Fold {i}" for i in range(1, 6)],
                   "datasets": [
                       {"label": name, "data": cv_results[name]["fold_aucs"],
                        "backgroundColor": VIVID_PALETTE[1] if name == CHAMPION_NAME else VIVID_PALETTE[4]}
                       for name in TOP2_NAMES
                   ]}

shap_view = {"key": "shap", "label": "SHAP Feature Importance",
             "labels": TOP_SHAP_FEATURES["feature"].tolist(),
             "datasets": [{"label": "Mean |SHAP|", "data": TOP_SHAP_FEATURES["mean_abs_shap"].round(4).tolist(),
                           "backgroundColor": _palette(len(TOP_SHAP_FEATURES))}]}

_lime_first_case = next(iter(_lime_cases.keys())) if _lime_cases else None
_lime_first_rows = [r for r in LIME_EXPLANATIONS if r["case"] == _lime_first_case] if _lime_first_case else []
lime_view = {"key": "lime", "label": f"LIME — {_lime_first_case}" if _lime_first_case else "LIME",
             "labels": [r["feature_condition"] for r in _lime_first_rows],
             "datasets": [{"label": "Local Weight", "data": [round(r["weight"], 4) for r in _lime_first_rows],
                           "backgroundColor": [VIVID_PALETTE[2] if r["weight"] >= 0 else VIVID_PALETTE[7]
                                                for r in _lime_first_rows]}]}

top_missing_5 = top_missing.head(5)
missing_view_15 = {"key": "top15", "label": f"Top {len(top_missing)} Columns",
                    "labels": top_missing["column"].tolist(),
                    "datasets": [{"label": "% Missing", "data": (top_missing["pct_null"] * 100).round(2).tolist(),
                                  "backgroundColor": _palette(len(top_missing))}]}
missing_view_5 = {"key": "top5", "label": "Top 5 Columns",
                   "labels": top_missing_5["column"].tolist(),
                   "datasets": [{"label": "% Missing", "data": (top_missing_5["pct_null"] * 100).round(2).tolist(),
                                 "backgroundColor": _palette(len(top_missing_5))}]}

exposure_view_dollars = {"key": "dollars", "label": "By Dollar Exposure",
                          "labels": ["Captured (TP)", "Missed (FN)", "False Alarm (FP)"],
                          "datasets": [{"data": [CAPTURED_DEFAULT_EXPOSURE, MISSED_DEFAULT_EXPOSURE, FALSE_ALARM_EXPOSURE],
                                        "backgroundColor": [VIVID_PALETTE[0], VIVID_PALETTE[7], VIVID_PALETTE[3]]}]}
exposure_view_count = {"key": "count", "label": "By Applicant Count",
                        "labels": ["Captured (TP)", "Missed (FN)", "False Alarm (FP)"],
                        "datasets": [{"data": [int(tp_n), int(fn_n), int(fp_n)],
                                      "backgroundColor": [VIVID_PALETTE[0], VIVID_PALETTE[7], VIVID_PALETTE[3]]}]}

SAMPLE_N = min(200, len(holdout_predictions_df))
sample_df = (
    holdout_predictions_df.sample(n=SAMPLE_N, random_state=SEED)
    .sort_values("predicted_probability", ascending=False)
    .round({"predicted_probability": 4, "amt_credit": 2})
)

html_path = build_html_dashboard(
    REPORTS_DIR / "notebook_01_dashboard.html",
    title="Problem 1 — Credit Default Prediction",
    subtitle=f"Champion: {CHAMPION_NAME} | Real holdout ROC-AUC: {metrics['roc_auc']:.4f} | {DEPLOYMENT_VERDICT}",
    kpi_cards=[
        {"label": "Champion Model", "value": CHAMPION_NAME},
        {"label": "Holdout ROC-AUC", "value": f"{metrics['roc_auc']:.4f}"},
        {"label": "Capture Rate", "value": f"{CAPTURE_RATE:.1%}"},
        {"label": "Captured Exposure", "value": f"${CAPTURED_DEFAULT_EXPOSURE:,.0f}"},
        {"label": "Real Default Rate", "value": f"{TARGET_RATE:.2%}"},
        {"label": "Integrity Checks", "value": f"{sum(1 for _, ok in checks if ok)}/{len(checks)} PASS"},
        {"label": "Top SHAP Driver", "value": TOP_SHAP_FEATURES.iloc[0]["feature"]},
    ],
    insights=INSIGHTS,
    charts=[
        {"id": "modelChart", "title": "Top-2 5-Fold CV Champion Selection", "type": "bar",
         "labels": model_view_cv["labels"], "datasets": model_view_cv["datasets"], "showLegend": False,
         "views": [model_view_cv, model_view_screen], "story": STORY_MODEL_CHART},
        {"id": "cvReportChart", "title": "5-Fold CV Report (Top 2 Models, Per-Fold Real ROC-AUC)", "type": "bar",
         "labels": cv_report_view["labels"], "datasets": cv_report_view["datasets"], "showLegend": True},
        {"id": "shapChart", "title": f"SHAP Feature Importance — Champion ({CHAMPION_NAME})", "type": "bar",
         "labels": shap_view["labels"], "datasets": shap_view["datasets"], "showLegend": False,
         "story": STORY_SHAP_CHART},
        {"id": "limeChart", "title": f"LIME — {_lime_first_case or 'Champion'} (real holdout case)", "type": "bar",
         "labels": lime_view["labels"], "datasets": lime_view["datasets"], "showLegend": False,
         "story": STORY_LIME_CHART},
        {"id": "missingChart", "title": "Top Columns by Real Missing-Value %", "type": "bar",
         "labels": missing_view_15["labels"], "datasets": missing_view_15["datasets"], "showLegend": False,
         "views": [missing_view_15, missing_view_5], "story": STORY_MISSING_CHART},
        {"id": "calibChart", "title": "Calibration by Score Decile (real holdout)", "type": "line",
         "labels": [str(int(d)) for d in calib_summary["decile"]],
         "datasets": [
             {"label": "Mean Predicted", "data": calib_summary["mean_predicted"].round(4).tolist(), "borderColor": VIVID_PALETTE[0], "backgroundColor": VIVID_PALETTE[0], "fill": False},
             {"label": "Actual Default Rate", "data": calib_summary["actual_rate"].round(4).tolist(), "borderColor": VIVID_PALETTE[7], "backgroundColor": VIVID_PALETTE[7], "fill": False},
         ],
         "note": f"Mean calibration gap: {MEAN_CALIBRATION_GAP:.4f}", "story": STORY_CALIB_CHART},
        {"id": "exposureChart", "title": "Holdout Credit Exposure by Outcome (real AMT_CREDIT)", "type": "doughnut",
         "labels": exposure_view_dollars["labels"], "datasets": exposure_view_dollars["datasets"],
         "views": [exposure_view_dollars, exposure_view_count], "story": STORY_EXPOSURE_CHART},
        {"id": "targetChart", "title": "Real Target Class Balance", "type": "doughnut",
         "labels": ["No Default (0)", "Default (1)"],
         "datasets": [{"data": target_counts["count"].tolist(), "backgroundColor": ["#0ca30c", "#d03b3b"]}],
         "story": STORY_TARGET_CHART},
    ],
    data_table={
        "title": f"Sampled Real Holdout Predictions ({SAMPLE_N} of {len(holdout_predictions_df):,} rows)",
        "columns": ["SK_ID_CURR", "actual_target", "predicted_probability", "predicted_label", "amt_credit", "outcome"],
        "rows": sample_df[["SK_ID_CURR", "actual_target", "predicted_probability", "predicted_label", "amt_credit", "outcome"]].values.tolist(),
        "filter_column": "outcome",
    },
)
print(f"[REPORTING] Real reporting package written: reports/{word_path.name}, reports/{excel_path.name}, "
      f"reports/{html_path.name}, plus {len(csv_paths)} CSV file(s) (all under decision_engine/reports/).")

# ---------------------------------------------------------------------------
# SECTION 14 — Save artifacts + governance stamp (SOP Stage 6: Production
# Packaging & Governance) — idempotent: overwrite in place, fixed paths
# ---------------------------------------------------------------------------
summary = {
    "notebook": "01_credit_default_prediction",
    "mega_project": "Mega Project 1 - Intelligent Underwriting & Automated Credit Decisioning",
    "problem": "Problem 1 - Credit Default Prediction",
    "random_seed": SEED,
    "n_rows_train_raw": N_APP_RAW,
    "n_rows_bureau_raw": N_BUREAU_RAW,
    "real_default_rate": TARGET_RATE,
    "feature_count": len(FEATURE_COLS),
    "eda_data_quality": {
        "n_columns_with_missing": int(len(null_counts)),
        "top_5_missing_pct": {row["column"]: round(float(row["pct_null"]), 4) for _, row in null_counts.head(5).iterrows()},
        "days_employed_anomaly_count": N_DAYS_EMPLOYED_ANOMALY,
        "days_employed_anomaly_pct": PCT_DAYS_EMPLOYED_ANOMALY,
        "iqr_outliers": OUTLIER_SUMMARY,
        "top_target_correlations": target_corr.abs().sort_values(ascending=False).head(3).round(4).to_dict(),
        "eda_chart_files": [p.name for p in EDA_CHART_PATHS],
    },
    "performance_config": {
        "logical_cores_detected": TOTAL_THREADS,
        "total_ram_gb_detected": TOTAL_RAM_GB,
        "cpu_thread_ceiling_applied": CPU_CEILING_THREADS,
        "ram_ceiling_gb": RAM_CEILING_GB,
        "cpu_affinity_pinned_cores": PERF.get("logical_cores"),
        "parquet_cache_dir": str(PARQUET_CACHE_DIR.name),
    },
    "model_selection": {
        "candidate_models_screened": CANDIDATE_NAMES,
        "screening_results": screening_results,
        "top2_advanced_to_cv": TOP2_NAMES,
        "champion_model": CHAMPION_NAME,
        "runner_up_model": RUNNER_UP_NAME,
        "cv_results": cv_results,
    },
    "explainability": {
        "shap_sample_size": SHAP_SAMPLE_N,
        "shap_top_10_features": shap_importance_df.head(10).round(4).to_dict("records"),
        "lime_cases_explained": list(_lime_cases.keys()),
        "lime_explanation_count": len(LIME_EXPLANATIONS),
    },
    "cv_results": cv_results,
    "champion_model": CHAMPION_NAME,
    "holdout_metrics": metrics,
    "feature_set_accuracy_comparison": {
        "v1_feature_set": "2-table: application_train + bureau (" + str(len(NUMERIC_FEATURES_V1 + CATEGORICAL_FEATURES_V1)) + " features)",
        "v2_feature_set": "7-table: + bureau_balance, previous_application, POS_CASH_balance, "
                          "installments_payments, credit_card_balance (" + str(len(FEATURE_COLS)) + " features)",
        "same_champion_architecture": CHAMPION_NAME,
        "same_holdout_row_count": int(len(X_holdout_v1)),
        "v1_real_holdout_auc": BASELINE_V1_AUC,
        "v2_real_holdout_auc": V2_AUC,
        "real_auc_improvement": AUC_IMPROVEMENT,
        "note": "Both AUCs measured on the identical real holdout rows (aligned by SK_ID_CURR), with the "
                "same champion model architecture and the same random seed -- the only variable is which "
                "feature set was used, so this delta is directly attributable to the richer data, not to a "
                "different model or a different split.",
    },
    "statistical_validation": {
        "bootstrap_resamples": N_BOOTSTRAP,
        "holdout_auc_ci_95": [AUC_CI_LOW, AUC_CI_HIGH],
        "mean_calibration_gap": MEAN_CALIBRATION_GAP,
        "split_half_psi": SPLIT_HALF_PSI,
        "deployment_checks": {name: ok for name, ok in deployment_checks},
        "failed_deployment_checks": _failed_deployment_checks,
        "deployment_verdict": DEPLOYMENT_VERDICT,
        "note": "deployment_checks (statistical robustness) is a separate check family from "
                "integrity_checks (structural pipeline sanity) below -- see deployment_verdict "
                "for which specific statistical check(s), if any, failed on this run.",
    },
    "financial_impact": {
        "total_holdout_exposure_usd": TOTAL_HOLDOUT_EXPOSURE,
        "captured_default_exposure_usd": CAPTURED_DEFAULT_EXPOSURE,
        "missed_default_exposure_usd": MISSED_DEFAULT_EXPOSURE,
        "false_alarm_exposure_usd": FALSE_ALARM_EXPOSURE,
        "capture_rate": CAPTURE_RATE,
        "assumptions": ASSUMPTIONS,
        "estimated_loss_prevented_usd_illustrative": ESTIMATED_LOSS_PREVENTED,
    },
    "integrity_checks": {name: bool(ok) for name, ok in checks},
    "reporting_artifacts": ["notebook_01_report.docx", "notebook_01_workbook.xlsx",
                             "notebook_01_dashboard.html"] + [f"{stem}.csv" for stem in csv_paths],
    "sop_stage_reached": "6 - Production Packaging & Governance",
    "runtime_seconds": round(time.time() - T0, 1),
}
with open(ARTIFACTS_DIR / "notebook_01_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

import joblib
joblib.dump(
    {"model": champion, "ordinal_encoder": ord_enc, "imputer": imputer,
     "feature_cols": FEATURE_COLS, "numeric_features": NUMERIC_FEATURES,
     "categorical_features": CATEGORICAL_FEATURES, "champion_name": CHAMPION_NAME},
    ARTIFACTS_DIR / "notebook_01_champion_model.joblib",
)

print(f"[DONE] Notebook 01 complete in {summary['runtime_seconds']}s using a {CPU_CEILING_THREADS}-thread "
      f"WARP ceiling. Champion={CHAMPION_NAME} (of top-2 {TOP2_NAMES} from a {len(CANDIDATE_NAMES)}-model "
      f"screen), holdout ROC-AUC={metrics['roc_auc']:.4f}. "
      f"Deployment verdict: {DEPLOYMENT_VERDICT}. "
      f"Artifacts written to decision_engine/artifacts/, reporting package written to "
      f"decision_engine/reports/ (both idempotent overwrite).")
